In [1]:
import pandas as pd
import numpy as np

import jax
import jaxlib
import jax.numpy as jnp
import numpyro
from numpyro import distributions as dist
from numpyro.handlers import seed, trace

from cntmosaic.datasets import load_age_distribution, load_template_patterns
from cntmosaic.sim import ParticipantGenerator, ContactMatrixGenerator, ContactGenerator
from cntmosaic.models.dists import IGMRF1D, IGMRF2D
from cntmosaic.models._utils import gmrf_adjacency_matrix
from cntmosaic.models._inference import run_inference_mcmc

import matplotlib.pyplot as plt

In [2]:
age_dist = load_age_distribution("United_States")
patterns = load_template_patterns("United_States")

df_part = ParticipantGenerator(age_dist.P.values).generate(1000, 0)
cint_matrix = ContactMatrixGenerator(patterns, age_dist.P.values).generate(max_margin=10, seed=0)
df_cnt = ContactGenerator(df_part, cint_matrix).generate(0)

In [ ]:
class vdKassteele:
  def __init__(self,
               part: pd.DataFrame,
               cnt: pd.DataFrame):
    self._validate_inputs(part, cnt)
    self.part = part.copy()
    self.cnt = cnt.copy()
    self._load()
    
  def _validate_inputs(self, part: pd.DataFrame, cnt: pd.DataFrame):
    part_cols = part.columns
    cnt_cols = cnt.columns
    
    if "id" not in part_cols:
      raise ValueError("participant DataFrame must contain 'id' column")
    if "id" not in cnt_cols:
      raise ValueError("contact DataFrame must containt 'id' column")
    
    if "age_part" not in part_cols:
      raise ValueError("participant DataFrame must contain 'age_part' column")
    if "age_cnt" not in cnt_cols:
      raise ValueError("contact DataFrame must contain 'age_cnt' column")
    
    if "y" not in cnt_cols:
      raise ValueError("Missing column 'y' in contact DataFrame")
    
  def _load(self):
    # [Do] Merge contact data and participant data
    self.merged = pd.merge(self.cnt, self.part, on="id", how="left")
    self.merged = (
			self.merged
			.groupby(["age_part", "age_cnt"], observed=True)
			.agg(y=("y", "count"))
			.reset_index()
		)
    
    self.N = self.merged["id"].nunique()
    self.y = jnp.array(self.merged["y"].values)
    
    self.C = self.merged["age_grp_cnt"].nunique()
    self.D = self.merged["age_grp_part"].nunique()
    self.cid = jnp.array(self.merged["age_grp_part"].cat.codes)
    self.did = jnp.array(self.merged["age_grp_cnt"].cat.codes)
    
    # self.adj_matrix = gmrf_adjacency_matrix(self.C, self.D, 4)
    
  def model(self):
    beta0 = numpyro.sample("baseline", dist.Normal(0., 10.))
    with numpyro.plate('random_effects', self.N):
      log_sigma = numpyro.sample("log_sigma", dist.Normal(0, 1.))
    
    beta = numpyro.sample("beta", IGMRF2D([self.C, self.D], 1, [1, 1]))
    log_cint = numpyro.deterministic('log_cint', beta0 + beta.reshape(self.C, self.D, order='F'))
    
    mu = jnp.exp(log_cint[self.cid, self.did] + log_sigma[self.iid])
    
    with numpyro.plate("data", len(self.y)):
      numpyro.sample("obs", dist.Poisson(rate=mu), obs=self.y)
      
  def print_model_shape(self):
    """Print the shapes of the model parameters."""
    tr = trace(seed(self.model, jax.random.PRNGKey(0))).get_trace()
    print(numpyro.util.format_shapes(tr))
    
  def run_inference_mcmc(
    self,
    rng_key,
    num_samples: int = 500,
    num_warmup: int = 500,
    num_chains: int = 2,
    **kwargs):
    """Run full Bayesian inference using Hamiltonian Monte Carlo and NUT Sampler.

    Parameters
    ----------
    rng_key:
      Random number generator key.
    num_samples: int, default=1000
      Number of samples to draw from the posterior.
    num_warmup: int, default=1000
      Number of warmup steps.
    num_chains: int, default=1
      Number of chains to run.
    **kwargs
      Additional keyword arguments to pass to the MCMC
    """
    if not isinstance(rng_key, jaxlib.xla_extension.ArrayImpl):
      rng_key = jax.random.PRNGKey(int(rng_key))
    self.mcmc = run_inference_mcmc(
      rng_key,
      self.model,
      num_samples=num_samples,
      num_warmup=num_warmup,
      num_chains=num_chains,
      **kwargs
    )